# Notebook 1 — Data (Breast + ST)

**What this notebook does**
1. Reads the HEST-1k sample table and picks the breast sections from one study named *Integrating spatial gene expression and breast tumour morphology via deep learning* (Breast + Spatial Transcriptomics, 23 patients, 68 sections, all with oncotree code IDC).
2. Downloads each section's **patches** (`.h5`, the model input) and **expression** (`.h5ad`, the target) to the data folder.
3. Matches every patch to its expression row by **barcode**.
4. Keeps only the genes that **every** section measured, and applies **log1p**.
5. Saves one **spot table** in which every spot carries its section, patient, barcode and coordinates.

**What it does not do:** pick the top 50 genes, scale or split anything. Those steps learn from the data, so they happen in Notebook 3, after the split, using training spots only.

**Where it runs:** Google Colab. Data is saved to the shared team folder `MyDrive/BDD_BJMJ/BDD_HEST` on Google Drive, so everyone on the team uses the same files.

**Before you run it**
- Runtime: **CPU** is enough (In colab: select Runtime → Change runtime type → CPU). No GPU needed.
- The shared folder `BDD_JJBM` must be visible in your My Drive. June owns it; everyone else opens **Shared with me**, right-clicks `BDD_JJBM` → **Organize → Add shortcut → My Drive**.
- A Hugging Face account with access to `MahmoodLab/hest` (accept the dataset terms on its Hugging Face page once).
- Drive space for the raw files (they count against the folder owner's storage). The test run (3 patients) is small; check the size printed in Step 2 before running all 23 patients.

**What it saves** (in `DATA_DIR`)

Paths are relative to `DATA_DIR` in google drive.

| File | Contents | Used by |
|---|---|---|
| `raw/patches/<id>.h5` | 224×224 H&E patch per spot | Notebook 2 |
| `raw/st/<id>.h5ad` | Original expression table (raw counts) | kept as the original |
| `expression/<id>.npz` | log1p expression, spots × shared genes (sparse) | Notebook 3 |
| `spot_table.csv` | One row per spot: `sample_id`, `patient`, `barcode`, `x`, `y`, `patch_index`, `row_in_sample` | Notebooks 2 and 3 |
| `genes.txt` | Shared gene names, same order as the expression columns | Notebook 3 |
| `samples.csv` | One row per section with its metadata | Notebooks 2 and 3 |

**To scale up:** change only the Settings cell (`N_PATIENTS = None` for all 23 patients). Nothing below it needs editing.

In [2]:
%pip install -q huggingface_hub anndata h5py scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 976.5 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 37.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.3/405.3 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [1]:
import os
import numpy as np
import pandas as pd
import h5py
import anndata
import scipy.sparse
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, login

ModuleNotFoundError: No module named 'anndata'

**Connect Google Drive:** pick the Google account that can see `BDD_BJMJ`.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Settings

Everything you might change is in this cell. Nothing below it should need editing.

- `TEAM_FOLDER`: the shared team folder on Drive
- `DATA_DIR`: where this notebook saves its data, inside the team folder
- `N_PATIENTS`: how many patients to use. `3` for the test run, `None` for all 23
- `STUDY_TITLE_START`: which study to use (the start of its `dataset_title` in the HEST table)

In [9]:
TEAM_FOLDER = "/content/drive/MyDrive/BDD_BJMJ"
DATA_DIR = os.path.join(TEAM_FOLDER, "HEST_breast_ST")

METADATA_FILE = "HEST_v1_1_0.csv"
ORGAN = "Breast"
SPECIES = "Homo sapiens"
TECHNOLOGY = "Spatial Transcriptomics"
STUDY_TITLE_START = "Integrating spatial gene expression and breast tumour morphology"

N_PATIENTS = 3

In [10]:
if not os.path.isdir(TEAM_FOLDER):
    raise FileNotFoundError(
        f"Can't find {TEAM_FOLDER}. Open Google Drive → Shared with me, right-click BDD_BJMJ → "
        "Organize → Add shortcut → My Drive, then run the Drive cell and this cell again."
    )

RAW_DIR = os.path.join(DATA_DIR, "raw")
EXPRESSION_DIR = os.path.join(DATA_DIR, "expression")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(EXPRESSION_DIR, exist_ok=True)

print("Saving to:", DATA_DIR)

Saving to: /content/drive/MyDrive/BDD_BJMJ/HEST_breast_ST


**Hugging Face login:** paste your token into the box that appears. Never write it in the code or upload it to GitHub.

In [13]:
login()

## Step 1. Choose the sections

The HEST table has one row per section. We keep the human breast sections from our study, and drop any without a patient ID (this study has none missing, but other studies do).

In this study the patient IDs look like `BC23209`, and each patient has 2–3 sections. Several sections from one patient are why the patient split matters: they must all land on the same side of the split.

In [ ]:
metadata_path = hf_hub_download(
    repo_id="MahmoodLab/hest",
    repo_type="dataset",
    filename=METADATA_FILE,
    local_dir=RAW_DIR,
)
metadata = pd.read_csv(metadata_path, encoding="utf-8-sig")
print("All samples in HEST:", len(metadata))

is_organ = metadata["organ"] == ORGAN
is_species = metadata["species"] == SPECIES
is_technology = metadata["st_technology"] == TECHNOLOGY
is_study = metadata["dataset_title"].str.startswith(STUDY_TITLE_START)
has_patient = metadata["patient"].notna()

study_samples = metadata[is_organ & is_species & is_technology & is_study & has_patient].copy()

print("Sections in our study:", len(study_samples))
print("Patients in our study:", study_samples["patient"].nunique())

In [ ]:
sections_per_patient = study_samples.groupby("patient")["id"].apply(list)
print(sections_per_patient.to_string())

For the test run we keep the first `N_PATIENTS` patients (sorted by ID, so everyone gets the same ones) with **all** of their sections.

In [ ]:
all_patients = sorted(study_samples["patient"].unique())

if N_PATIENTS is None:
    chosen_patients = all_patients
else:
    chosen_patients = all_patients[:N_PATIENTS]

samples = study_samples[study_samples["patient"].isin(chosen_patients)]
samples = samples.sort_values(["patient", "id"]).reset_index(drop=True)
sample_ids = samples["id"].tolist()

print("Patients used:", len(chosen_patients), chosen_patients)
print("Sections used:", len(sample_ids), sample_ids)
print("Spots expected (from metadata):", samples["spots_under_tissue"].sum())

## Step 2. Download patches and expression

For each section we download two files:
- `patches/<id>.h5`: one 224×224 H&E patch per spot, plus its barcode and coordinates
- `st/<id>.h5ad`: the expression table, spots × genes, with each spot's barcode

Files that are already downloaded are skipped, so if the run stops (or Colab disconnects) you can just run this cell again.

In [ ]:
for sample_id in sample_ids:
    patches_path = os.path.join(RAW_DIR, "patches", f"{sample_id}.h5")
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")

    if os.path.exists(patches_path) and os.path.exists(st_path):
        print(sample_id, "already downloaded")
        continue

    hf_hub_download(repo_id="MahmoodLab/hest", repo_type="dataset",
                    filename=f"patches/{sample_id}.h5", local_dir=RAW_DIR)
    hf_hub_download(repo_id="MahmoodLab/hest", repo_type="dataset",
                    filename=f"st/{sample_id}.h5ad", local_dir=RAW_DIR)
    print(sample_id, "downloaded")

In [ ]:
total_bytes = 0
for sample_id in sample_ids:
    total_bytes += os.path.getsize(os.path.join(RAW_DIR, "patches", f"{sample_id}.h5"))
    total_bytes += os.path.getsize(os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad"))

size_per_section = total_bytes / len(sample_ids)
print(f"Downloaded: {total_bytes / 1e9:.2f} GB for {len(sample_ids)} sections")
print(f"Estimate for all {len(study_samples)} sections: {size_per_section * len(study_samples) / 1e9:.1f} GB")

## Step 3. Match patches to expression by barcode

The barcode is the spot's ID. A patch and an expression row belong to the same spot only if their barcodes match, so we never line them up by position.

For each section we keep the spots that have **both** a patch and an expression row. We also record:
- `patch_index`: where the spot's patch sits in the `.h5` file (Notebook 2 reads patches in this order)
- `row_in_sample`: the spot's row in this section's saved expression file

In [ ]:
def barcode_to_text(barcode):
    if hasattr(barcode, "ravel"):
        barcode = barcode.ravel()[0]
    if isinstance(barcode, bytes):
        barcode = barcode.decode("utf-8")
    return str(barcode)


def read_patch_barcodes(patches_path):
    with h5py.File(patches_path, "r") as f:
        raw_barcodes = f["barcode"][:]
        coords = f["coords"][:]
    barcodes = np.array([barcode_to_text(b) for b in raw_barcodes])
    return barcodes, coords

In [ ]:
spot_tables = []
gene_lists = []

for sample_id in sample_ids:
    patches_path = os.path.join(RAW_DIR, "patches", f"{sample_id}.h5")
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")

    patch_barcodes, coords = read_patch_barcodes(patches_path)

    adata = anndata.read_h5ad(st_path, backed="r")
    expression_barcodes = set(adata.obs_names)
    gene_names = pd.Index(adata.var_names)
    gene_lists.append(list(anndata.utils.make_index_unique(gene_names)))
    adata.file.close()

    keep = np.array([b in expression_barcodes for b in patch_barcodes])
    patient = samples.loc[samples["id"] == sample_id, "patient"].iloc[0]

    table = pd.DataFrame({
        "sample_id": sample_id,
        "patient": patient,
        "barcode": patch_barcodes[keep],
        "x": coords[keep, 0],
        "y": coords[keep, 1],
        "patch_index": np.where(keep)[0],
    })
    table["row_in_sample"] = np.arange(len(table))
    spot_tables.append(table)

    print(f"{sample_id}: {len(patch_barcodes)} patches, {keep.sum()} matched to expression")

spot_table = pd.concat(spot_tables, ignore_index=True)
print()
print("Total spots:", len(spot_table))

## Step 4. Keep the genes every section measured

Different sections can list slightly different genes. Notebook 3 needs the same columns for every spot, so we keep only genes present in **all** chosen sections.

This is not gene selection: it doesn't look at expression values, only at which genes exist, so it can be done before the split. Choosing the top 50 genes by variance happens in Notebook 3, on training spots only.

In [ ]:
shared_genes = set(gene_lists[0])
for genes in gene_lists[1:]:
    shared_genes = shared_genes & set(genes)

shared_genes = sorted(shared_genes)

print("Genes per section:", [len(g) for g in gene_lists])
print("Genes shared by all sections:", len(shared_genes))

## Step 5. Save log1p expression for each section

For each section we:
1. reorder its expression rows to match the spot table (by barcode)
2. keep the shared genes, in the same order for every section
3. apply log1p, so ln(1 + count): this evens out very large counts and keeps zeros at 0
4. save it as a sparse matrix (most counts are 0, so this keeps the files small)

log1p is applied to each number on its own and learns nothing from other spots, so doing it here can't leak test information.

In [ ]:
for sample_id in sample_ids:
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")
    adata = anndata.read_h5ad(st_path)
    adata.var_names_make_unique()

    sample_spots = spot_table[spot_table["sample_id"] == sample_id]
    adata = adata[sample_spots["barcode"].values, shared_genes]

    counts = scipy.sparse.csr_matrix(adata.X, dtype=np.float32)
    log_expression = counts.log1p()

    scipy.sparse.save_npz(os.path.join(EXPRESSION_DIR, f"{sample_id}.npz"), log_expression)
    print(f"{sample_id}: saved {log_expression.shape[0]} spots x {log_expression.shape[1]} genes")

In [ ]:
spot_table.to_csv(os.path.join(DATA_DIR, "spot_table.csv"), index=False)

with open(os.path.join(DATA_DIR, "genes.txt"), "w") as f:
    for gene in shared_genes:
        f.write(gene + "\n")

samples.to_csv(os.path.join(DATA_DIR, "samples.csv"), index=False)

print("Saved spot_table.csv, genes.txt and samples.csv to", DATA_DIR)

## Step 6. Check the result

Quick checks before moving on:
- every section has spots, and the counts match the metadata roughly
- each patient's sections are listed together
- one section plotted on the slide, coloured by total expression, should look like tissue (not random noise). If it looks random, the barcode matching went wrong.

In [ ]:
summary = spot_table.groupby(["patient", "sample_id"]).size().rename("spots").reset_index()
summary = summary.merge(samples[["id", "spots_under_tissue"]], left_on="sample_id", right_on="id")
summary = summary.drop(columns="id").rename(columns={"spots_under_tissue": "spots_in_metadata"})
print(summary.to_string(index=False))
print()
print("Patients:", spot_table["patient"].nunique())
print("Sections:", spot_table["sample_id"].nunique())
print("Spots:", len(spot_table))
print("Genes:", len(shared_genes))

In [ ]:
check_id = sample_ids[0]
check_spots = spot_table[spot_table["sample_id"] == check_id]
check_expression = scipy.sparse.load_npz(os.path.join(EXPRESSION_DIR, f"{check_id}.npz"))
total_log_expression = np.asarray(check_expression.sum(axis=1)).ravel()

plt.figure(figsize=(5, 5))
plt.scatter(check_spots["x"], check_spots["y"], c=total_log_expression, s=20, cmap="viridis")
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.colorbar(label="sum of log1p expression")
plt.title(f"{check_id}: spots on the slide")
plt.show()

## Next

Run **Notebook 2 (Embedding)** in Colab on a **GPU** runtime. It reads `spot_table.csv` and the patch files, and saves one embedding file per section in the same spot order.